[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-03-datasets-hooks-checkpoints-tracking.ipynb)

# Advanced Discussion 3 — Dataset and batch size, hooks, resumable checkpoints, a trustworthy experiment record

Reproduces the chapter 3 advanced discussion: a real Dataset class and the batch-size trade-off, forward hooks and anomaly mode, checkpoint/resume equivalence, and logging a data fingerprint and code version to MLflow.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: uses scikit-learn's bundled digits and a temporary local MLflow database.

In [ ]:
%pip install -q mlflow

In [ ]:
import os, io, time, hashlib, subprocess, tempfile, json
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

## 1. a real Dataset class and what batch size does to speed and accuracy

In [ ]:
class TabularDataset(Dataset):
    """Wraps feature/label arrays; the place to put per-item transforms, not the training loop."""
    def __init__(self, X, y): self.X, self.y = torch.tensor(X, dtype=torch.float32), torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]
X, y = load_digits(return_X_y=True); X = X / 16.0
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
def train_once(batch_size, epochs=15, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    dl = DataLoader(TabularDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    model = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 10)); opt = torch.optim.Adam(model.parameters(), lr=lr)
    t0 = time.perf_counter(); steps = 0
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step(); steps += 1
    dt = time.perf_counter() - t0
    with torch.no_grad(): acc = (model(torch.tensor(Xva, dtype=torch.float32)).argmax(1) == torch.tensor(yva)).float().mean().item()
    return dt, steps, acc
print("batch  seconds  steps  val_acc  (15 epochs, same lr)")
for bs in (8, 32, 128, 512):
    dt, steps, acc = train_once(bs); print("%5d  %7.2f  %5d  %.3f" % (bs, dt, steps, acc))

## 2. debugging with hooks: shapes, activation statistics, gradient norms, NaN hunting

In [ ]:
model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
log = []
def make_hook(name):
    def hook(mod, inp, out): log.append((name, tuple(inp[0].shape), tuple(out.shape), round(out.abs().mean().item(), 3)))
    return hook
handles = [m.register_forward_hook(make_hook(f"{i}:{type(m).__name__}")) for i, m in enumerate(model)]
xb = torch.tensor(Xtr[:16], dtype=torch.float32); F.cross_entropy(model(xb), torch.tensor(ytr[:16])).backward()
for h in handles: h.remove()
for row in log: print("  ", row)
print("  grad norms:", {n: round(p.grad.norm().item(), 3) for n, p in model.named_parameters()})
bad = nn.Sequential(nn.Linear(64, 8), nn.ReLU(), nn.Linear(8, 1))
xb_bad = torch.tensor(Xtr[:4], dtype=torch.float32); xb_bad[0, 0] = float("nan")
try:
    with torch.autograd.set_detect_anomaly(True):
        bad(xb_bad).sum().backward()
    print("  anomaly mode: no exception (forward NaN just propagates)")
except Exception as e:
    print("  anomaly mode raised:", type(e).__name__)
print("  isfinite check on output:", bool(torch.isfinite(bad(xb_bad)).all()))

## 3. checkpoint + resume gives the same result as an uninterrupted run

In [ ]:
def make_model_opt(seed=0):
    torch.manual_seed(seed); m = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)); return m, torch.optim.Adam(m.parameters(), lr=2e-3)
def run_epochs(m, o, start, end, seed_base=100):
    for ep in range(start, end):
        g = torch.Generator().manual_seed(seed_base + ep)                   # per-epoch shuffling seed -> resumable
        dl = DataLoader(TabularDataset(Xtr, ytr), batch_size=64, shuffle=True, generator=g)
        for xb, yb in dl: o.zero_grad(); F.cross_entropy(m(xb), yb).backward(); o.step()
def save(path, m, o, epoch): torch.save({"model": m.state_dict(), "opt": o.state_dict(), "epoch": epoch}, path)
tmp = tempfile.mkdtemp(); ck = os.path.join(tmp, "ck.pt")
m1, o1 = make_model_opt(); run_epochs(m1, o1, 0, 10)                        # uninterrupted
m2, o2 = make_model_opt(); run_epochs(m2, o2, 0, 4); save(ck, m2, o2, 4)      # interrupted after epoch 4
m3, o3 = make_model_opt(seed=999); st = torch.load(ck); m3.load_state_dict(st["model"]); o3.load_state_dict(st["opt"]); run_epochs(m3, o3, st["epoch"], 10)
same = all(torch.allclose(a, b) for a, b in zip(m1.state_dict().values(), m3.state_dict().values()))
print("resumed run identical to uninterrupted run:", same)
m4, o4 = make_model_opt(seed=999); st = torch.load(ck); m4.load_state_dict(st["model"]); run_epochs(m4, o4, st["epoch"], 10)         # forgot the optimizer state
print("resume WITHOUT optimizer state identical:", all(torch.allclose(a, b) for a, b in zip(m1.state_dict().values(), m4.state_dict().values())))

## 4. what to log: parameters, metrics, data fingerprint, code version, environment

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///" + os.path.join(tmp, "mlflow.db").replace(chr(92), "/")); mlflow.set_experiment("orbit-churn-demo")
def data_fingerprint(*arrays):
    h = hashlib.sha256()
    for a in arrays: h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]
def git_sha():
    try: return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception: return "unknown"
with mlflow.start_run(run_name="baseline-mlp"):
    mlflow.log_params({"lr": 2e-3, "batch_size": 64, "hidden": 64, "seed": 0})
    mlflow.set_tags({"git_sha": git_sha(), "data_fingerprint": data_fingerprint(Xtr, ytr, Xva, yva), "torch": torch.__version__, "python": os.sys.version.split()[0]})
    mlflow.log_metric("val_acc", 0.97, step=10)
    mlflow.log_dict({"classes": list(range(10)), "preprocessing": "pixels / 16"}, "model_card.json")
run = mlflow.search_runs(experiment_names=["orbit-churn-demo"]).iloc[0]
print("logged tags:", {k.replace("tags.", ""): run[k] for k in run.index if k.startswith("tags.") and "mlflow" not in k})

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.